In [4]:
import PyPDF2
import re
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('punkt')

# -------------------------------------------------------
# 1. EXTRACT TEXT FROM ANY PDF
# -------------------------------------------------------
def extract_pdf_text(path):
    reader = PyPDF2.PdfReader(path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + " "
    return text

pdf_path = input("Enter PDF filename (example: file.pdf): ")
document_text = extract_pdf_text(pdf_path)

# -------------------------------------------------------
# 2. CLEAN & CHUNK PDF TEXT
# -------------------------------------------------------
document_text = re.sub(r'\s+', ' ', document_text).strip()

sentences = sent_tokenize(document_text)

if len(sentences) < 5:
    print("ERROR: The PDF does not contain enough text to train a model.")
    exit()

# -------------------------------------------------------
# 3. AUTO-GENERATE TRAINING DATA (Q–A PAIRS)
# -------------------------------------------------------
questions = []
answers = []

for sentence in sentences:
    cleaned = sentence.strip()
    if len(cleaned.split()) > 3:
        questions.append(f"What about: {cleaned[:50]}?")
        answers.append(cleaned)

# -------------------------------------------------------
# 4. TRAIN-TEST SPLIT
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    questions, answers, test_size=0.25, random_state=42
)

# -------------------------------------------------------
# 5. VECTORIZE (TF-IDF)
# -------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vectors = vectorizer.fit_transform(X_train)

# -------------------------------------------------------
# 6. QA FUNCTION (COSINE SIMILARITY)
# -------------------------------------------------------
def answer_question(query):
    query_vector = vectorizer.transform([query])
    sims = cosine_similarity(query_vector, X_train_vectors).flatten()
    idx = sims.argmax()
    return y_train[idx]

# -------------------------------------------------------
# 7. MODEL EVALUATION
# -------------------------------------------------------
preds = [answer_question(q) for q in X_test]
accuracy = accuracy_score(y_test, preds)

print("\nMODEL ACCURACY:", accuracy)
print("\nPDF loaded and QA system is ready.\n")

# -------------------------------------------------------
# 8. INTERACTIVE QUESTION ANSWERING
# -------------------------------------------------------
print("Ask anything from the PDF (type 'exit' to stop):\n")

while True:
    q = input("Your Question: ")
    if q.lower() == "exit":
        break
    print("Answer:", answer_question(q))
    print()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Enter PDF filename (example: file.pdf): FINAL REPORT.PDF


C:\Users\user\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")



MODEL ACCURACY: 0.0

PDF loaded and QA system is ready.

Ask anything from the PDF (type 'exit' to stop):

Your Question: what is HTML5
Answer: 1 Thymeleaf Logo 19 2 Bootstrap 5 Logo 21 3 HTML5 & CSS3 Logo 22 4 JavaScript Logo 24 5 Spring Boot Logo 25 6 Spring Data JPA Logo 29 7 MySQL with XAMPP Logo 31 8 Apache Tomcat Logo 33 9 Landing Page - Hero Section 34 10 Landing Page - Features and Footer 34 11 Login Choice Page 35 12 User Registration Page - Form View 36 13 User Registration Page - Complete View 36 14 User Login Page 37 15 Meal Kits Browse Page - Grid View 38 16 Meal Kits Browse Page - Subscription Modal 38 17 User Dashboard - Active Subscriptions 39 18 Admin Login Page 40 19 Admin Dashboard - Statistics View 41 20 Manage Meal Kits - Admin Panel 42 CHEF'S PANTRY - MEAL KIT SUBSCRIPTION SYSTEM 2025 Department of CSE, GMRIT 1 CHAPTER 1 INTRODUCTION 1.1 Introduction The internship program serves as a crucial bridge between academic learning and professional practice, providing s

In [3]:
pip install sentence-transformers PyPDF2 nltk


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import re
import PyPDF2
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('punkt', quiet=True)

# -------------------------------------------------------
# 1. PDF Extraction
# -------------------------------------------------------
def extract_pdf_text(path):
    reader = PyPDF2.PdfReader(path)
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + " "
    return text.strip()


# -------------------------------------------------------
# 2. Clean & split into sentences
# -------------------------------------------------------
def preprocess_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    sentences = sent_tokenize(text)

    return [s.strip() for s in sentences if len(s.split()) >= 4]


# -------------------------------------------------------
# 3. Improved QA Model (TF-IDF + Keyword Boosting)
# -------------------------------------------------------
class PDF_QA:
    def __init__(self, sentences):
        self.sentences = sentences
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform(sentences)

    def keyword_score(self, query, sentence):
        """
        Adds a small boost to similarity based on shared keywords.
        This replaces fuzzywuzzy (no external modules required).
        """
        q_words = set(query.lower().split())
        s_words = set(sentence.lower().split())
        common = q_words.intersection(s_words)
        return len(common) / (len(q_words) + 1)

    def answer(self, query):
        if not query or len(query.split()) < 2:
            return "Please ask a more detailed question."

        # TF-IDF similarity
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.matrix).flatten()

        # Keyword boosting
        boosted_scores = []
        for i, sent in enumerate(self.sentences):
            boosted = sims[i] + self.keyword_score(query, sent) * 0.25
            boosted_scores.append(boosted)

        best_idx = boosted_scores.index(max(boosted_scores))
        best_score = boosted_scores[best_idx]

        if best_score < 0.10:
            return "I could not find that information in the PDF."

        return self.sentences[best_idx]

    def evaluate(self, X_test, y_test):
        preds = [self.answer(q) for q in X_test]
        return accuracy_score(y_test, preds)


# -------------------------------------------------------
# 4. Main Program
# -------------------------------------------------------
def main():
    pdf_path = input("Enter PDF filename: ").strip()

    if not os.path.exists(pdf_path):
        print("ERROR: File not found:", pdf_path)
        return

    raw = extract_pdf_text(pdf_path)
    if not raw:
        print("ERROR: No text detected in PDF.")
        return

    sentences = preprocess_text(raw)
    if len(sentences) < 5:
        print("ERROR: Not enough meaningful sentences.")
        return

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        sentences, sentences, test_size=0.25, random_state=42
    )

    # Build model
    model = PDF_QA(sentences)

    # Evaluate
    acc = model.evaluate(X_test, y_test)
    print(f"\nMODEL ACCURACY: {acc:.4f}\n")

    print("PDF loaded successfully. Ask questions (type 'exit' to quit)\n")

    # QA Loop
    while True:
        q = input("Your Question: ").strip()
        if q.lower() == "exit":
            print("Exiting.")
            break

        print("Answer:", model.answer(q), "\n")


if __name__ == "__main__":
    main()


Enter PDF filename: FINAL REPORT.pdf

MODEL ACCURACY: 1.0000

PDF loaded successfully. Ask questions (type 'exit' to quit)

Your Question: what is the tools and technologies used
Answer: 2.5 Technologies and Tools ByteXL specializes in training and development using:  Backend Technologies: Java, Spring Boot, Spring MVC, Hibernate, Node.js  Frontend Technologies: HTML5, CSS3, JavaScript, React, Angular, Thymeleaf  Databases: MySQL, PostgreSQL, MongoDB, Oracle  Cloud Platforms: AWS, Azure, Google Cloud Platform  Development Tools: Eclipse, IntelliJ IDEA, Visual Studio Code, Git, Maven  DevOps: Docker, Kubernetes, Jenkins, CI/CD pipelines CHEF'S PANTRY - MEAL KIT SUBSCRIPTION SYSTEM 2025 Department of CSE, GMRIT 6 CHAPTER-3 TASKS TAKEN UP AND PROBLEM DEFINTION 3.1 Introduction The food industry has witnessed a significant transformation with the rise of meal-kit delivery services, offering convenience and quality to consumers. 

Your Question: what is the project name
Answer: The Ch

Your Question: what is html5
Answer: Implementation in Chef's Pantry:  Responsive navigation bar with dropdown menus  Card-based meal kit display layout  Modal dialogs for subscription forms  Form styling with validation feedback  Alert messages for user notifications  Button styling with hover effects  Responsive grid for dashboard statistics HTML5 & CSS3 HTML5 is the latest evolution of the standard markup language for structuring web content. 

Your Question: what is javascript
Answer: It is designed to process and create HTML, XML, JavaScript, CSS, and even plain text. 

Your Question: exit
Exiting.
